# CS 195: Natural Language Processing
## Recurrent Neural Networks and Language Modeling

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ericmanley/s26-CS195NLP/blob/main/F5_3_RNNLanguageModels.ipynb)


## References

- [Chapter 3: N-gram Language Models](https://web.stanford.edu/~jurafsky/slp3/3.pdf). *Speech and Language Processing.* Daniel Jurafsky & James H. Martin
- [Chapter 13: RNNs and LSTMs](https://web.stanford.edu/~jurafsky/slp3/13.pdf). *Speech and Language Processing.* Daniel Jurafsky & James H. Martin
- [PyTorch `nn.RNN` docs](https://pytorch.org/docs/stable/generated/torch.nn.RNN.html)
- [PyTorch `nn.Embedding` docs](https://pytorch.org/docs/stable/generated/torch.nn.Embedding.html)


In [ ]:
#import sys
#!{sys.executable} -m pip install torch transformers

## Neural Language Modeling

Today we will build a simple token-level language model with a *recurrent* neural network.

A **language model** is a kind of classification model:
- the **classes** are all possible tokens in the vocabulary
- at each time step, the model predicts **which token comes next** - this is called a **causal language model**

This is just a fancier model for doing the same task we did with Markov Models!

Here's what this might look like if we did it with a non-recurrent network:

<div>
    <img src="images/neural_language_modeling.png">
</div>

Image from an old version of https://web.stanford.edu/~jurafsky/slp3/

## Why Recurrent Models?

In our earlier models, we pooled the text into one fixed vector.
That worked for some tasks, but it threw away a lot of information about word order.

A recurrent neural network processes a sequence one token at a time.
It keeps a **hidden state** that gets updated after each token, so it can carry information from earlier parts of the sequence forward through time.




## Causal Language Modeling as Repeated Classification

Suppose we have this token sequence:

`[the, cat, sat, on, the, mat]`

A causal language model learns training pairs like:
- input: `[the]` target: `cat`
- input: `[the, cat]` target: `sat`
- input: `[the, cat, sat]` target: `on`

At each step, the model outputs a vector of logits over the whole vocabulary.
So the learning task is still classification, but now we do it over and over across the sequence.


## Visualizing the RNN

A **recurrent neural network** is a neural network with a loop inside of it - some of the outputs in one layer become inputs of the same or an earlier layer

<div>
    <img src="images/RNN_highlevel.png" width=250>
</div>

* $x_{t}$: neural network input at time $t$
* $h_{t}$: hidden layer state at time $t$
* $y_{t}$: output layer state at time $t$

*Allows information from past inputs to affect current predictions*

At a high level, an RNN has a loop: information from one step feeds into the next step. In this image, the inputs are shown on bottom and the outputs on top

<div>
    <img src="images/RNN_as_feedforward.png" width=420>
</div>

* $h_{t-1}$: hidden layer state at time $t-1$ is an input to $h_{t}$

Another way to see it is as a feedforward network that has been repeated over time with the **same weights reused at every step**.
That weight sharing is what makes it possible to process sequences of different lengths.

image credits: https://web.stanford.edu/~jurafsky/slp3/13.pdf


## Unrolling the Network Through Time


Later outputs continue to be influenced by the entire sequence

<div>
    <img src="images/RNN_unroll.png" width=520>
</div>

image credit: https://web.stanford.edu/~jurafsky/slp3/13.pdf

When we **unroll** an RNN, we draw one copy of the recurrent cell for each time step.
This makes it easier to see that:
- the input at time $t$ affects the hidden state at time $t$
- the hidden state at time $t$ becomes part of the input at time $t+1$
- later predictions are influenced by everything that came before

This is the central promise of recurrent models: *instead of treating a text as an unordered bag, they build up a running representation as they read the sequence*.


In [1]:
import random
import torch
import torch.nn as nn
import torch.optim as optim
from transformers import AutoTokenizer

random.seed(42)
torch.manual_seed(42)


## A Tiny Training Corpus

We will start with a very small corpus just so the model can train quickly and we can inspect what it is doing.
This is not enough data for a good language model, but it is enough to see the workflow.


In [2]:
sentences = [
    "dogs and cats are common household pets",
    "cats and dogs both need food and water",
    "my dog and my cat play together",
    "cats and dogs can live in the same home",
    "the puppy and kitten were adopted together",
    "sharks and whales live in the ocean",
    "fish swim in the ocean",
    "boats travel across the ocean",
    "neural networks learn patterns from data",
    "transformers use attention to model language",
    "retrieval systems rank documents by relevance",
    "embedding vectors can capture semantic similarity"
]

for s in sentences:
    print(s)


dogs and cats are common household pets
cats and dogs both need food and water
my dog and my cat play together
cats and dogs can live in the same home
the puppy and kitten were adopted together
sharks and whales live in the ocean
fish swim in the ocean
boats travel across the ocean
neural networks learn patterns from data
transformers use attention to model language
retrieval systems rank documents by relevance
embedding vectors can capture semantic similarity


## Use a Pretrained Hugging Face Tokenizer

Today, we will reuse the tokenizer from the `distilbert-base-uncased` model. It's a big, but managable vocabulary to work with.



In [3]:
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

vocab_size = tokenizer.vocab_size
print("vocab size:", vocab_size)

example = tokenizer(sentences[0], add_special_tokens=False)
print(tokenizer.convert_ids_to_tokens(example["input_ids"]))
print(example["input_ids"])


vocab size: 30522
['dogs', 'and', 'cats', 'are', 'common', 'household', 'pets']
[6077, 1998, 8870, 2024, 2691, 4398, 18551]


## Build Causal LM Training Examples

We will use a sliding window.
For each position, the input is a short sequence of previous token IDs and the target is the next token ID.

For example, if the window size is 4 and the token sequence is:
`[5, 8, 12, 7, 3]`

then one training example might be:
- input: `[5, 8, 12, 7]`
- target: `3`


### Worked Example of the Training Task

Suppose a tokenized sentence begins like this:

`dogs and cats are common household pets`

Then the model might see training examples like:
- input: `[PAD, PAD, PAD, dogs]` target: `and`
- input: `[PAD, PAD, dogs, and]` target: `cats`
- input: `[PAD, dogs, and, cats]` target: `are`
- input: `[dogs, and, cats, are]` target: `common`

So each example asks the model: **given this short prefix, which token should come next?**

That is why this is still a classification problem: the target is one correct token ID from the full vocabulary.


In [4]:
window_size = 4
pad_id = tokenizer.pad_token_id

def make_lm_examples(sentences, tokenizer, window_size=4):
    X = []
    y = []

    for sentence in sentences:
        ids = tokenizer(sentence, add_special_tokens=False)["input_ids"]
        for i in range(1, len(ids)):
            # Grab the recent context before the target token
            prefix = ids[max(0, i - window_size):i]
            # Add [PAD] tokens on the left if the context is shorter than the window
            prefix = [pad_id] * (window_size - len(prefix)) + prefix
            X.append(prefix)
            y.append(ids[i])

    return torch.tensor(X, dtype=torch.long), torch.tensor(y, dtype=torch.long)

X_train, y_train = make_lm_examples(sentences, tokenizer, window_size=window_size)
print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)
print("example input ids:", X_train[0])
print("example target id:", y_train[0].item())


X_train shape: torch.Size([69, 4])
y_train shape: torch.Size([69])
example input ids: tensor([   0,    0,    0, 6077])
example target id: 1998


## Build a Simple RNN Language Model

The model has three main parts:
- an `Embedding` layer to turn token IDs into vectors
- an `RNN` layer to process the sequence left-to-right
- a `Linear` layer to turn the final hidden state into logits over the vocabulary

We use the final hidden state because it summarizes the prefix we gave the model.


### Language Modeling View of the Architecture

<div>
    <img src="images/RNN_languagemodeling.png" width=700>
</div>

image credits: https://web.stanford.edu/~jurafsky/slp3/13.pdf





This diagram lines up closely with the code we are using:
- the `Embedding` layer converts each token ID into a dense vector
- the RNN reads those vectors left-to-right and updates its hidden state at each step
- the final hidden state stands for "what the model currently knows about the prefix"
- the `Linear` layer converts that state into logits over all possible next tokens
- 
It's common in PyTorch to create a class to represent your model that inherits from `nn.Module`
* define the `forward` function to show how the feed-forward step is computed

In [5]:
class TinyRNNLM(nn.Module):
    def __init__(self, vocab_size, embedding_dim=32, hidden_dim=64):
        super().__init__()

        # Look up a learned vector for each token id
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=pad_id)

        # Read the sequence left-to-right, updating a hidden state at each step
        # batch_first=True shapes the data like this [batch_size, sequence_length, embedding_dim]
        self.rnn = nn.RNN(embedding_dim, hidden_dim, batch_first=True)

        # Turn the final hidden state into scores for every token in the vocabulary
        self.output = nn.Linear(hidden_dim, vocab_size)

    def forward(self, input_ids):
        # input_ids shape: [batch_size, sequence_length]
        emb = self.embedding(input_ids)

        # emb shape: [batch_size, sequence_length, embedding_dim]
        # hidden shape: [1, batch_size, hidden_dim]
        rnn_out, hidden = self.rnn(emb)

        # Remove the extra first dimension since we only have one RNN layer
        last_hidden = hidden.squeeze(0)

        # logits shape: [batch_size, vocab_size]
        logits = self.output(last_hidden)
        return logits
model = TinyRNNLM(vocab_size)
model


TinyRNNLM(
  (embedding): Embedding(30522, 32, padding_idx=0)
  (rnn): RNN(32, 64, batch_first=True)
  (output): Linear(in_features=64, out_features=30522, bias=True)
)

## Train the Model

Notice that this is still the same familiar pattern:
- forward pass
- cross-entropy loss
- backward pass
- optimizer step

The only real difference is that our labels are token IDs instead of sentiment labels or class IDs for whole documents.


In [6]:
loss_fn = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

# set the model to be in training mode
model.train()

for epoch in range(500):
    logits = model(X_train)
    loss = loss_fn(logits, y_train)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if (epoch + 1) % 100 == 0:
        predictions = logits.argmax(dim=1)
        acc = (predictions == y_train).float().mean().item()
        print(f"epoch {epoch+1}, loss={loss.item():.4f}, acc={acc:.4f}")


epoch 100, loss=0.0269, acc=0.9855
epoch 200, loss=0.0228, acc=0.9855
epoch 300, loss=0.0216, acc=0.9855
epoch 400, loss=0.0211, acc=0.9855
epoch 500, loss=0.0208, acc=0.9855


## Generate Text from the Model

Generation means:
1. give the model a starting prefix
2. predict the next token
3. append that token to the prefix
4. repeat

This is the same core idea that modern causal language models use, even though their architectures are much more powerful.


In [7]:
unk_id = tokenizer.unk_token_id

def generate_text(start_text, model, tokenizer, num_tokens=8, window_size=4):

    # we're not training, so set the model to eval mode
    model.eval()
    generated_ids = tokenizer(start_text, add_special_tokens=False)["input_ids"]

    for _ in range(num_tokens):
        # Grab the recent context before the target token
        prefix = generated_ids[-window_size:]
        # Add [PAD] tokens on the left if the context is shorter than the window
        prefix = [pad_id] * (window_size - len(prefix)) + prefix
        x = torch.tensor([prefix], dtype=torch.long)

        with torch.no_grad():
            logits = model(x)
            next_id = logits.argmax(dim=1).item()

        generated_ids.append(next_id)

    tokens = tokenizer.convert_ids_to_tokens([i for i in generated_ids if i != pad_id and i != unk_id])
    return tokens

print(generate_text("cats and", model, tokenizer))
print(generate_text("neural", model, tokenizer))
print(generate_text("the ocean", model, tokenizer))


['cats', 'and', 'dogs', 'both', 'need', 'food', 'and', 'water', 'can', 'live']
['neural', 'networks', 'learn', 'patterns', 'from', 'data', 'learn', 'patterns', 'from']
['the', 'ocean', 'the', 'ocean', 'home', 'networks', 'learn', 'patterns', 'from', 'data']


## Group Discussion

Once we use a bigger training set, this will be computationally infeasible

Assume each of the output logits takes 4 bytes to store

How many bytes does it take to store the output for all output tokens in the vocabulary?

What do you think is a practical limit for the number of training examples we can use with the code as-is?

## Batching 

PyTorch has functions that allow you to train in smaller batches


In [ ]:
from torch.utils.data import TensorDataset, DataLoader

dataset = TensorDataset(X_train, y_train)
loader = DataLoader(dataset, batch_size=128, shuffle=True)

Then you can update your training loop to go through each batch within each epoch

In [ ]:
for epoch in range(50):
    # We'll accumulate statistics across all batches so we can report one loss/accuracy per epoch
    total_loss = 0.0
    total_correct = 0
    total_examples = 0

    # New idea: instead of pushing all training examples through the model at once,
    # the DataLoader gives us one small batch at a time
    for X_batch, y_batch in loader:

        # This part is the same as before, just on one batch instead of all of X_train / y_train
        logits = model(X_batch)
        loss = loss_fn(logits, y_batch)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # Keep track of how this batch did so we can average over the whole epoch
        batch_size = X_batch.size(0) # X_batch has shape [batch_size, sequence_length], so this gives us the batch_size
        total_loss += loss.item() * batch_size
        predictions = logits.argmax(dim=1)
        total_correct += (predictions == y_batch).sum().item()
        total_examples += batch_size

    # Turn the accumulated totals into epoch-level averages
    avg_loss = total_loss / total_examples
    acc = total_correct / total_examples
    print(f"epoch {epoch+1}, loss={avg_loss:.4f}, acc={acc:.4f}")

## Exercise

Update the code above to use this new dataloader and training loop. Test it with different batch sizes and make sure it is working.


## Making it work with an Accelerator

So far, we haven't told PyTorch to use the GPU or other accelerator. Here's a standard patter for determining if we have an accelerator we can use

In [ ]:
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"Using device: {device}")

then, every time you create a model, you have to load it onto the accelerator device with a line like

In [ ]:
model.to(device)

and, every time you create an input tensor, you need to tell it to put that on the device too like this

In [ ]:
X_train = X_train.to(device)
y_train = y_train.to(device)

## Exercise

Go into your code above and set it to use an accelerator. But, make sure not to move the entire dataset to the device like this! Do it at the batch level.

Try to find all of the places in the code (i.e., `X_batch`, `y_batch`, etc.) where you need to explicitly mode the tensors to the accelerator device

## Applied Exploration

Use your new version of the code (with batching and `.to(device)`) with a new, larger dataset. I suggest https://huggingface.co/datasets/sarnab/Shakespeare_Corpus

Experiment with the metaparameters
1. window size
2. embedding dimension
3. hidden dimension
4. number of training epochs (500 is probably too many with the larger data set!)

Show some examples of new text you generated for different values, and try to determine what values tend to give you good results


## What Are the Limitations of This Model?

This notebook is enough to show the core causal LM idea, but plain RNNs have important limits:
- it is hard for them to remember information from far back in the sequence
- they process tokens one step at a time, so computation is inherently sequential
- training can become unstable on longer sequences

This is where LSTMs and GRUs enter the story: they are recurrent models designed to preserve information better across time.
Later, attention will give a different solution to the same general problem.


## LSTM and GRU

One of the main problems with plain RNNs is that it is hard for training signals and useful information to survive across many time steps.
This is related to the **vanishing gradient** problem.

<div>
    <img src="images/vanishing_gradient.png" width=520>
</div>

Historically, this led to gated recurrent models like LSTMs and GRUs.
They add mechanisms for deciding what information to keep, update, or forget.

<div>
    <img src="images/RNN-vs-LSTM-vs-GRU.png" width=650>
</div>

<div>
    <img src="images/LSTM_node.png" width=650>
</div>

I don't want to spend a whole day experimenting with these, but you can feel free to peform some experiments with them as an Extended Implementation Creative Synthesis project.

Things to keep in mind about them, though
- plain RNNs introduced sequence-aware modeling
- LSTMs and GRUs improved long-term memory

image credits: https://web.stanford.edu/~jurafsky/slp3